In [1]:
import FinanceDataReader as fdr

In [2]:
import numpy as np
import pandas as pd

In [3]:
import datetime
from dateutil.relativedelta import relativedelta

In [4]:
# n개월 전 날짜 계산 함수(개월단위)
def calculate_start_date(months_ago, end_date):
    start_date = datetime.datetime.strptime(end_date, '%Y-%m-%d') - relativedelta(months=months_ago)
    return start_date.strftime('%Y-%m-%d')

# 오늘 날짜 구하기
today = datetime.datetime.today()
today_str = today.strftime('%Y-%m-%d')

In [5]:
kospi = fdr.StockListing('KOSPI')
kosdaq = fdr.StockListing('KOSDAQ')
etfs = fdr.StockListing('ETF/KR')

In [6]:
kospi_code = kospi['Code'].to_list()

In [7]:
kosdaq_code = kosdaq['Code'].to_list()

In [8]:
etf_code = etfs['Symbol'].to_list()

In [9]:
all_stock_codes = kospi_code + kosdaq_code + etf_code

In [10]:
month3 = calculate_start_date(3, today_str)

In [65]:
month2 = calculate_start_date(2, today_str)

In [14]:
all_data = []

In [15]:
# 조회 실패한 종목 리스트
failed_codes = []

In [16]:
for code in all_stock_codes:
    try:
        # DataReader 실행
        close_df = fdr.DataReader(code, month3)[['Close']].reset_index()
        close_df['Code'] = code
        all_data.append(close_df)
    
    except Exception:
        failed_codes.append(code)  # 조회 실패한 종목 저장

In [17]:
# 조회 실패한 종목 출력
if failed_codes:
    print("\n조회 실패한 종목 코드 목록:")
    print(", ".join(failed_codes))  # 한 줄로 출력
else:
    print("\n모든 종목 데이터 조회 성공!")



조회 실패한 종목 코드 목록:
0001S0, 0007F0, 0008S0, 0000Y0, 0005A0, 0000J0, 0016X0, 0000D0, 0008T0, 0010E0, 0015F0, 0008E0, 0005D0, 0001P0, 0000Z0, 0005C0, 0007N0, 0015E0, 0013R0, 0013P0, 0007G0, 0015K0, 0005G0, 0004G0


In [18]:
# 모든 데이터를 하나의 DataFrame으로 변환
result_df = pd.concat(all_data, ignore_index=True)

# 최종 데이터 확인
print("\n수집된 데이터 일부:")
print(result_df.head())


수집된 데이터 일부:
        Date  Close    Code
0 2024-11-25  57900  005930
1 2024-11-26  58300  005930
2 2024-11-27  56300  005930
3 2024-11-28  55500  005930
4 2024-11-29  54200  005930


In [19]:
import cx_Oracle

In [25]:
dsn = cx_Oracle.makedsn("localhost", 1521, service_name="xe")
conn = cx_Oracle.connect(user="c##PROJECT", password="k5002", dsn=dsn)
cursor = conn.cursor()

In [21]:
# Date 형식 변환 (YYYY-MM-DD → YY/MM/DD)
result_df["TRADE_DATE"] = pd.to_datetime(result_df["Date"]).dt.strftime("%y/%m/%d")

In [26]:
# STK_CODE → STK_ID 매핑을 위한 조회 쿼리
stk_code_to_id = {}
cursor.execute("SELECT STK_CODE, STK_ID FROM MKT_SEC_STK")
for stk_code, stk_id in cursor.fetchall():
    stk_code_to_id[stk_code] = stk_id

In [27]:
# DataFrame을 데이터베이스에 삽입하기 위한 리스트 변환
insert_data = []
for _, row in result_df.iterrows():
    stk_id = stk_code_to_id.get(row["Code"])  # STK_ID 조회
    if stk_id is None:
        print(f"⚠️ STK_ID를 찾을 수 없음: {row['Code']}")  # 매핑되지 않은 코드 출력
        continue

    insert_data.append((stk_id, row["Close"], row["TRADE_DATE"]))

In [28]:
# 데이터 삽입 쿼리 실행 (REC_STK_ID는 시퀀스 사용)
insert_sql = """
INSERT INTO REC_STK (REC_STK_ID, STK_ID, REC_PRICE, TRADE_DATE, CDATE) 
VALUES (REC_STK_SEQ.NEXTVAL, :1, :2, TO_DATE(:3, 'YY/MM/DD'), SYSDATE)
"""

In [29]:
# executemany()로 한 번에 삽입 (성능 최적화)
cursor.executemany(insert_sql, insert_data)
conn.commit()

In [30]:
cursor.close()
conn.close()